- Ative a máquina virtual com:

`source nomeambiente/bin/activate`


- Instalando pacotes e bibliotecas necessárias

In [3]:
%pip install langchain
%pip install langchain-community
%pip install -U langchain-core
%pip install -qU "langchain[openai]"
%pip install langchain-openai


from langchain_community.document_loaders import CSVLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from langchain_core.documents import Document
import csv
import getpass
import os

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Carrega o csv
caminho_arquivo = "./dados_relevantes.csv"

documentos_resultados = []
coluna_id = 'researcher_id'
colunas_com_textos_longos = ['abstract','articles','description_project','project_name']
# Incluí mais colunas que vão ser divididas em mais de uma chunk

# Tentei imitar o Semantic Chunker com RecursiveCharacterTextSplliter
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

# Personalizei a ordem dos separadores como achei melhor
text_splitter = RecursiveCharacterTextSplitter (
    separators=["\n\n", "\n", "; ", ". ", ", "],
    chunk_size = CHUNK_SIZE,
    chunk_overlap= CHUNK_OVERLAP,
    length_function=len,
)

with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo_csv:
    # dictReader vai ler cada linha como um dicionário (coluna -> valor)
    leitor_csv = csv.DictReader(arquivo_csv)
    
    # itera sobre cada linha do csv
    for i, linha in enumerate(leitor_csv):
        id = linha[coluna_id]
        
        for nome_coluna, valor_coluna in linha.items():
            
            if valor_coluna == 'Sem registro': # Eliminei os chunks das colunas sem registro
                continue
            elif nome_coluna in colunas_com_textos_longos:  
                chunks_de_texto = text_splitter.split_text(valor_coluna)
              
                for j, chunk in enumerate(chunks_de_texto):
                    conteudo = f"Um trecho da coluna '{nome_coluna}' é {chunk}" # Analisar se fica melhor contextualizando com nome da coluna mesmo <----
                    metadata = {
                        'source': caminho_arquivo,
                        'reseacher_id': id,
                        'row': i,
                        'column': nome_coluna,
                        'chunk_index': j    
                    }
                    doc = Document(page_content=conteudo, metadata=metadata)
                    documentos_resultados.append(doc)
            else:
                if nome_coluna != coluna_id: # para não repetir linha
                    conteudo = f"A informação de '{nome_coluna}' é: '{valor_coluna}'"
                    metadata = {
                        'source': caminho_arquivo,
                        'reseacher_id': id,
                        'row': i,
                        'column': nome_coluna
                    }
                    doc = Document(page_content=conteudo, metadata=metadata)
                    documentos_resultados.append(doc)

print(f"\nFinalizou. Total de {len(documentos_resultados)} documentos (chunks) gerados.")
print("-" * 50)

# --- Visualização de Resultados ---
print("Visualizar alguns dos chunks gerados para ver a diferença:\n")

for doc in documentos_resultados[:40]:
    e_chunk_de_texto_longo = 'chunk_index' in doc.metadata
    
    if e_chunk_de_texto_longo:
        print("\x1b[36m--- Chunk de Texto Longo ---\x1b[0m") # Azul
    else:
        print("\x1b[32m--- Chunk de Coluna Curta ---\x1b[0m") # Verde

    print(f"Conteúdo: {doc.page_content}")
    print(f"Tem {len(doc.page_content)} caracteres")
    print(f"Metadados: {doc.metadata}\n")


Finalizou. Total de 8842 documentos (chunks) gerados.
--------------------------------------------------
Visualizar alguns dos chunks gerados para ver a diferença:

--- Chunk de Coluna Curta ---
Conteúdo: Do documento com ID '00d79509-2e3a-4ef9-95f6-f9d5c79c34cf', a informação de 'researcher_name' é: 'Hugo Saba Pereira Cardoso'
Tem 124 caracteres
Metadados: {'source': './dados_relevantes.csv', 'reseacher_id': '00d79509-2e3a-4ef9-95f6-f9d5c79c34cf', 'row': 0, 'column': 'researcher_name'}

--- Chunk de Texto Longo ---
Conteúdo: Do documento com ID '00d79509-2e3a-4ef9-95f6-f9d5c79c34cf', um trecho da coluna 'abstract' é Hugo Saba Pereira Cardoso é um pesquisador com vínculo principal na Campus Integrado de Manufatura e Tecnologias, especializado na área de Ciências Sociais Aplicadas. Sua trajetória acadêmica inclui uma sólida formação, com destaque para a obtenção do título de Doutorado em Difusão do Conhecimento pela Universidade Federal da Bahia, concluído entre os anos de 2009 e 2013.